# Práctica 1: Saturación del Uplink en un Campus en Hora Punta

**Objetivo:** Trasladar el problema clásico de capacidad uplink con control rápido de potencia a una situación moderna en la que decenas de estudiantes usan simultáneamente el móvil para subir audios, imágenes, tareas al LMS, videollamadas cortas y sincronización en la nube.

**Escenario:** En el cambio de clase, una zona exterior de campus concentra usuarios caminando a distintas velocidades. Se quiere saber cuántos terminales puede admitir la célula antes de acercarse al límite de carga del enlace ascendente.

**Conceptos reforzados:**
- Factor de actividad
- Control de potencia rápido
- Efecto de la velocidad sobre la carga
- Ganancia de procesado
- Carga uplink (fracción de polo)
- Margen por desvanecimiento rápido e interferencia intercelular

**Herramientas:** Python, Jupyter, NumPy, pandas, matplotlib

---
## 1. Resolución Teórica: Fórmulas y Conceptos

### 1.1 Ecuación de Carga Fraccional del Uplink

Para un sistema WCDMA/UMTS con control rápido de potencia, la carga fraccional del enlace ascendente se expresa como:

$$\eta_{UL} = N \cdot (1 + i) \cdot \frac{v \cdot \left(\frac{E_b}{N_0}\right) \cdot \frac{R_b}{W}}{1 + \left(\frac{E_b}{N_0}\right) \cdot \frac{R_b}{W}}$$

Donde:
| Símbolo | Descripción | Valor típico |
|---|---|---|
| $N$ | Número de usuarios simultáneos | variable |
| $i$ | Factor de interferencia intercelular | 0.55 – 0.80 |
| $W$ | Tasa de chips (ancho de banda de espectro ensanchado) | 3.84 Mchip/s |
| $R_b$ | Tasa de bits del servicio | 12.2 / 32 / 64 kb/s |
| $v$ | Factor de actividad (proporción de tiempo en transmisión) | 0.4 – 0.7 |
| $E_b/N_0$ | Relación energía por bit a densidad espectral de ruido requerida | depende de servicio y velocidad |

La fracción $W/R_b$ es la **ganancia de procesado** $G_p$:

$$G_p = \frac{W}{R_b}$$

### 1.2 Capacidad Máxima ($N_{max}$)

Despejando $N$ para un umbral de carga objetivo $\eta_{target}$:

$$N_{max} = \frac{\eta_{target}}{(1+i)} \cdot \frac{1 + \frac{E_b}{N_0} \cdot \frac{R_b}{W}}{v \cdot \frac{E_b}{N_0} \cdot \frac{R_b}{W}}$$

### 1.3 Efecto de la velocidad sobre $E_b/N_0$ requerida

A velocidades elevadas el control rápido de potencia no puede seguir el desvanecimiento de Rayleigh, lo que incrementa el $E_b/N_0$ requerido (margen por desvanecimiento rápido $\Delta_{ff}$):

$$\left(\frac{E_b}{N_0}\right)_{requerida}(v) = \left(\frac{E_b}{N_0}\right)_{ref} \cdot 10^{\Delta_{ff}(v)/10}$$

Valores de referencia para UMTS con AMD vocal (12.2 kb/s, BLER objetivo 1%):

| Velocidad | $\Delta_{ff}$ | Causa |
|---|---|---|
| 3 km/h | 0 dB | Control de potencia muy eficaz |
| 15 km/h | +1.5 dB | Ligera degradación |
| 50 km/h | +3.5 dB | Pérdida de seguimiento del PC |

In [ ]:
# ─────────────────────────────────────────────
# Importaciones
# ─────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from itertools import product

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

---
## 2. Parámetros del Sistema

In [ ]:
# ─────────────────────────────────────────────
# 2.1  Parámetros fijos del sistema
# ─────────────────────────────────────────────
W_chips = 3.84e6          # Tasa de chips WCDMA [chip/s]

# Perfiles de servicio: nombre, tasa de bits [b/s]
services = {
    'Voz AMR 12.2 kb/s': 12.2e3,
    'Datos  32 kb/s':    32e3,
    'Datos  64 kb/s':    64e3,
}

# Velocidades de usuario [km/h]
speeds_kmh = [3, 15, 50]

# Factores de actividad
activity_factors = np.arange(0.40, 0.75, 0.05)   # [0.40, 0.45, …, 0.70]

# Factor de interferencia intercelular i
i_factor = 0.65           # valor medio del rango 0.55 – 0.80

# Umbral de carga objetivo
eta_target = 0.70         # valor central del rango 0.65 – 0.75

print(f"Ancho de banda WCDMA : W = {W_chips/1e6:.2f} Mchip/s")
print(f"Factor intercelular  : i = {i_factor}")
print(f"Umbral de carga      : η_target = {eta_target}")
print()

# Ganancia de procesado por servicio
print("Ganancias de procesado:")
for name, rb in services.items():
    gp = W_chips / rb
    print(f"  {name:25s}  Gp = {gp:.1f}  ({10*np.log10(gp):.1f} dB)")

In [ ]:
# ─────────────────────────────────────────────
# 2.2  Eb/N0 requerido según servicio y velocidad
# ─────────────────────────────────────────────
#
# Valores de referencia (3 km/h) basados en simulaciones 3GPP TR 25.942:
#   Voz AMR 12.2 kb/s  → ~5.0 dB
#   Datos  32 kb/s     → ~4.0 dB
#   Datos  64 kb/s     → ~3.5 dB
#
# Margen por desvanecimiento rápido Δff (dB) cuando el control de
# potencia no puede seguir el canal:
#   3 km/h  → 0.0 dB  (PC muy eficaz, canal cuasi-estático)
#  15 km/h  → 1.5 dB  (degradación ligera)
#  50 km/h  → 3.5 dB  (PC pierde el seguimiento)

ebn0_ref_dB = {
    'Voz AMR 12.2 kb/s': 5.0,
    'Datos  32 kb/s':    4.0,
    'Datos  64 kb/s':    3.5,
}

delta_ff_dB = {3: 0.0, 15: 1.5, 50: 3.5}

def ebn0_linear(service_name, speed_kmh):
    """Devuelve Eb/N0 en escala lineal para un servicio y velocidad dados."""
    eb_n0_dB = ebn0_ref_dB[service_name] + delta_ff_dB[speed_kmh]
    return 10 ** (eb_n0_dB / 10)

print("Eb/N0 requerido [dB] por servicio y velocidad:")
header = "".join([f"{v:>10} km/h" for v in speeds_kmh])
print(f"{'Servicio':<25}  {header}")
print("-" * 65)
for sname in services:
    row = "".join([f"{ebn0_ref_dB[sname]+delta_ff_dB[v]:>14.1f} dB" for v in speeds_kmh])
    print(f"{sname:<25}  {row}")

---
## 3. Funciones de Cálculo

In [ ]:
def uplink_load(N, rb, v, ebn0_lin, W=W_chips, i=i_factor):
    """
    Carga fraccional del uplink para N usuarios con el mismo servicio.

    Parámetros
    ----------
    N        : int/float  – número de usuarios simultáneos
    rb       : float      – tasa de bits del servicio [b/s]
    v        : float      – factor de actividad [0, 1]
    ebn0_lin : float      – Eb/N0 requerido en escala lineal
    W        : float      – tasa de chips [chip/s]
    i        : float      – factor de interferencia intercelular

    Devuelve
    --------
    eta : float  – fracción de carga uplink [0, 1)
    """
    x = ebn0_lin * rb / W          # Eb/N0 × Rb/W (inversa de Gp efectiva)
    eta = N * (1 + i) * (v * x) / (1 + x)
    return eta


def nmax(rb, v, ebn0_lin, eta_tgt=eta_target, W=W_chips, i=i_factor):
    """
    Número máximo de usuarios antes de superar el umbral de carga.

    Parámetros
    ----------
    rb       : float – tasa de bits [b/s]
    v        : float – factor de actividad
    ebn0_lin : float – Eb/N0 requerido en escala lineal
    eta_tgt  : float – umbral de carga objetivo (por defecto eta_target)
    W        : float – tasa de chips [chip/s]
    i        : float – factor de interferencia intercelular

    Devuelve
    --------
    N_max : float – número máximo de usuarios (puede ser no entero)
    """
    x = ebn0_lin * rb / W
    N_max = (eta_tgt / (1 + i)) * (1 + x) / (v * x)
    return N_max


# ── Verificación rápida ──────────────────────────────────────────────────────
rb_test  = services['Voz AMR 12.2 kb/s']
ebn0_t   = ebn0_linear('Voz AMR 12.2 kb/s', 3)
v_test   = 0.50
N_test   = nmax(rb_test, v_test, ebn0_t)
eta_test = uplink_load(int(N_test), rb_test, v_test, ebn0_t)

print(f"Prueba (voz 12.2 kb/s, v=0.50, 3 km/h):")
print(f"  Nmax  = {N_test:.2f} usuarios")
print(f"  η con N=int(Nmax)={int(N_test)} → {eta_test:.4f}  (debe ser ≤ {eta_target})")

---
## 4. Nmax frente a Velocidad y Factor de Actividad

### 4.1 Tablas resumen

In [ ]:
# ─────────────────────────────────────────────
# Tabla: Nmax para v=0.50 (factor de actividad fijo)
# Filas = servicios, columnas = velocidades
# ─────────────────────────────────────────────
v_fixed = 0.50

records = []
for sname, rb in services.items():
    row = {'Servicio': sname}
    for spd in speeds_kmh:
        ebn0 = ebn0_linear(sname, spd)
        n = nmax(rb, v_fixed, ebn0)
        row[f'{spd} km/h'] = int(n)   # truncamos al entero inferior
    records.append(row)

df_speed = pd.DataFrame(records).set_index('Servicio')
print(f"Nmax (factor de actividad v = {v_fixed}, η_target = {eta_target}, i = {i_factor})")
display(df_speed)

In [ ]:
# ─────────────────────────────────────────────
# Tabla: Nmax para velocidad fija = 3 km/h
# Filas = servicios, columnas = actividad
# ─────────────────────────────────────────────
spd_fixed = 3   # km/h

act_cols = [round(a, 2) for a in activity_factors]
records2 = []
for sname, rb in services.items():
    ebn0 = ebn0_linear(sname, spd_fixed)
    row  = {'Servicio': sname}
    for a in act_cols:
        row[f'v={a:.2f}'] = int(nmax(rb, a, ebn0))
    records2.append(row)

df_act = pd.DataFrame(records2).set_index('Servicio')
print(f"Nmax ({spd_fixed} km/h, η_target = {eta_target}, i = {i_factor})")
display(df_act)

### 4.2 Curvas Nmax – Velocidad

In [ ]:
spd_range = np.array([1, 3, 5, 10, 15, 20, 30, 50, 70, 100])  # km/h

# Interpolamos Δff de forma continua (lineal en dB)
delta_ff_knots = np.array([[3, 0.0], [15, 1.5], [50, 3.5]])

def delta_ff_interp(v_kmh):
    """Interpolación/extrapolación lineal del margen de desvanecimiento [dB]."""
    return np.interp(v_kmh, delta_ff_knots[:, 0], delta_ff_knots[:, 1])

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=False)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

for ax, (sname, rb), color in zip(axes, services.items(), colors):
    for v_act, ls in [(0.40, '--'), (0.50, '-'), (0.70, ':')]:
        nm_vals = []
        for spd in spd_range:
            dff   = delta_ff_interp(spd)
            ebn0  = 10 ** ((ebn0_ref_dB[sname] + dff) / 10)
            nm_vals.append(nmax(rb, v_act, ebn0))
        ax.plot(spd_range, nm_vals, linestyle=ls, color=color,
                label=f'v={v_act:.2f}', linewidth=2)

    ax.set_title(sname, fontsize=11, fontweight='bold')
    ax.set_xlabel('Velocidad [km/h]')
    ax.set_ylabel('$N_{max}$ [usuarios]')
    ax.legend(title='Factor actividad', fontsize=9)
    ax.xaxis.set_minor_locator(mticker.AutoMinorLocator())
    ax.set_xlim(left=0)
    ax.set_ylim(bottom=0)

fig.suptitle(f'Curvas $N_{{max}}$ – Velocidad  (η_target={eta_target}, i={i_factor})',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('nmax_vs_velocidad.png', bbox_inches='tight')
plt.show()
print("Figura guardada: nmax_vs_velocidad.png")

### 4.3 Curvas Nmax – Factor de Actividad

In [ ]:
v_range = np.linspace(0.30, 0.90, 100)

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=False)

for ax, (sname, rb), color in zip(axes, services.items(), colors):
    for spd, ls in [(3, '-'), (15, '--'), (50, ':')]:
        ebn0 = ebn0_linear(sname, spd)
        nm_vals = [nmax(rb, v, ebn0) for v in v_range]
        ax.plot(v_range, nm_vals, linestyle=ls, color=color,
                label=f'{spd} km/h', linewidth=2)

    ax.set_title(sname, fontsize=11, fontweight='bold')
    ax.set_xlabel('Factor de actividad $v$')
    ax.set_ylabel('$N_{max}$ [usuarios]')
    ax.legend(title='Velocidad', fontsize=9)
    ax.set_xlim([0.30, 0.90])
    ax.set_ylim(bottom=0)

fig.suptitle(f'Curvas $N_{{max}}$ – Factor de Actividad  (η_target={eta_target}, i={i_factor})',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('nmax_vs_actividad.png', bbox_inches='tight')
plt.show()
print("Figura guardada: nmax_vs_actividad.png")

---
## 5. Comparativa: Control de Potencia Eficaz vs. Degradado

Evaluamos dos escenarios extremos:
- **PC eficaz**: Eb/N0 de referencia (3 km/h, sin margen adicional).
- **PC degradado**: Eb/N0 aumentado +3.5 dB (equivalente a 50 km/h o calibración imperfecta).

In [ ]:
scenarios = {
    'PC eficaz (3 km/h, Δff=0 dB)':      0.0,
    'PC ligero (15 km/h, Δff=+1.5 dB)':  1.5,
    'PC degradado (50 km/h, Δff=+3.5 dB)': 3.5,
}

v_act = 0.50

fig, ax = plt.subplots(figsize=(10, 5))

x_pos   = np.arange(len(services))
bar_w   = 0.22
palette = ['#2ca02c', '#ff7f0e', '#d62728']

for k, (sc_name, delta) in enumerate(scenarios.items()):
    heights = []
    for sname, rb in services.items():
        ebn0_dB  = ebn0_ref_dB[sname] + delta
        ebn0_lin = 10 ** (ebn0_dB / 10)
        heights.append(int(nmax(rb, v_act, ebn0_lin)))
    bars = ax.bar(x_pos + k * bar_w, heights, bar_w,
                  label=sc_name, color=palette[k], alpha=0.85, edgecolor='white')
    for bar, h in zip(bars, heights):
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.5, str(h),
                ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xticks(x_pos + bar_w)
ax.set_xticklabels(list(services.keys()), fontsize=10)
ax.set_ylabel('$N_{max}$ [usuarios]')
ax.set_title(f'Impacto del Control de Potencia sobre $N_{{max}}$\n'
             f'(v={v_act}, η_target={eta_target}, i={i_factor})',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.savefig('comparativa_pc.png', bbox_inches='tight')
plt.show()
print("Figura guardada: comparativa_pc.png")

In [ ]:
# ── Tabla de pérdida relativa de capacidad ───────────────────────────────────
rows_pc = []
for sc_name, delta in scenarios.items():
    row = {'Escenario PC': sc_name}
    for sname, rb in services.items():
        ebn0_lin = 10 ** ((ebn0_ref_dB[sname] + delta) / 10)
        row[sname] = int(nmax(rb, v_act, ebn0_lin))
    rows_pc.append(row)

df_pc = pd.DataFrame(rows_pc).set_index('Escenario PC')

# Pérdida porcentual respecto al mejor caso
df_pc_pct = (df_pc.iloc[0] - df_pc) / df_pc.iloc[0] * 100
df_pc_pct.index.name = 'Pérdida capacidad (%)'

print("Nmax por escenario de control de potencia:")
display(df_pc)
print("\nPérdida de capacidad respecto al caso PC eficaz:")
display(df_pc_pct.round(1))

---
## 6. Regla de Admisión Sencilla

### 6.1 Definición del algoritmo

La regla propuesta es **basada en carga uplink estimada**:

1. La estación base mantiene un estimador de carga uplink en tiempo real $\hat{\eta}_{UL}$.
2. Cuando llega una solicitud de nueva sesión con servicio de tasa $R_b$ y actividad esperada $v$:
   - Se calcula el incremento de carga que añadiría ese usuario: $\Delta\eta = \frac{(1+i) \cdot v \cdot x}{1+x}$, con $x = (E_b/N_0) \cdot R_b/W$.
   - Si $\hat{\eta}_{UL} + \Delta\eta \leq \eta_{target}$: **sesión admitida**.
   - Si $\hat{\eta}_{UL} + \Delta\eta > \eta_{target}$: **sesión rechazada** (redirigir o diferir).
3. Umbral operativo $\eta_{target} = 0.70$ (margen de seguridad 30% antes del polo de carga).

### 6.2 Simulación de secuencia de admisión

In [ ]:
def delta_load(rb, v, ebn0_lin, W=W_chips, i=i_factor):
    """Incremento de carga uplink por un usuario adicional."""
    x = ebn0_lin * rb / W
    return (1 + i) * (v * x) / (1 + x)


def admission_control_simulation(arrivals, eta_tgt=eta_target):
    """
    Simula la admisión/rechazo de una secuencia de solicitudes.

    arrivals : lista de dicts con claves 'service', 'speed_kmh', 'activity'
    Devuelve : lista de dicts con resultado de cada intento
    """
    current_load = 0.0
    active_sessions = []
    results = []

    for t, req in enumerate(arrivals):
        rb    = services[req['service']]
        spd   = req['speed_kmh']
        v     = req['activity']
        ebn0  = ebn0_linear(req['service'], spd)
        dl    = delta_load(rb, v, ebn0)

        if current_load + dl <= eta_tgt:
            admitted = True
            current_load += dl
            active_sessions.append(dl)
        else:
            admitted = False

        results.append({
            't': t + 1,
            'Servicio':  req['service'],
            'v. km/h':   spd,
            'Actividad': v,
            'Δη':        round(dl, 4),
            'η acum.':   round(current_load, 4),
            'Admitida':  '✔' if admitted else '✘',
        })

    return results, current_load


# ── Secuencia de ejemplo (mix realista en hora punta) ────────────────────────
np.random.seed(42)
service_names = list(services.keys())
n_arrivals    = 30

arrivals_example = [
    {
        'service':    np.random.choice(service_names, p=[0.4, 0.4, 0.2]),
        'speed_kmh':  np.random.choice(speeds_kmh,   p=[0.5, 0.35, 0.15]),
        'activity':   round(np.random.uniform(0.40, 0.70), 2),
    }
    for _ in range(n_arrivals)
]

results, final_load = admission_control_simulation(arrivals_example)
df_adm = pd.DataFrame(results).set_index('t')

admitted_count = sum(r['Admitida'] == '✔' for r in results)
rejected_count = n_arrivals - admitted_count

print(f"Sesiones admitidas  : {admitted_count}/{n_arrivals}")
print(f"Sesiones rechazadas : {rejected_count}/{n_arrivals}")
print(f"Carga final         : η = {final_load:.4f} (umbral = {eta_target})")
print()
display(df_adm)

In [ ]:
# ── Evolución de la carga durante la secuencia de admisión ───────────────────
carga_acum  = [r['η acum.'] for r in results]
admitidas   = [r['Admitida'] == '✔' for r in results]
indices_ok  = [i+1 for i, a in enumerate(admitidas) if a]
indices_ko  = [i+1 for i, a in enumerate(admitidas) if not a]

fig, ax = plt.subplots(figsize=(12, 4))
ax.step(range(1, n_arrivals+1), carga_acum, where='post',
        color='steelblue', linewidth=2, label='Carga acumulada η')
ax.axhline(eta_target, color='red', linestyle='--', linewidth=1.5,
           label=f'Umbral η_target = {eta_target}')

# Marcas de admitidas/rechazadas
for idx in indices_ok:
    ax.axvline(idx, color='green', alpha=0.15, linewidth=6)
for idx in indices_ko:
    ax.axvline(idx, color='red', alpha=0.25, linewidth=6)

ax.set_xlabel('Número de solicitud')
ax.set_ylabel('Carga uplink η')
ax.set_title('Evolución de la carga uplink con control de admisión\n'
             '(verde = admitida, rojo = rechazada)', fontweight='bold')
ax.set_xlim([1, n_arrivals])
ax.set_ylim([0, 1.0])
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('admision_carga.png', bbox_inches='tight')
plt.show()
print("Figura guardada: admision_carga.png")

---
## 7. Sensibilidad del Factor de Reutilización (i)

El factor intercelular $i$ varía según la geometría de la red. A continuación analizamos su efecto sobre $N_{max}$.

In [ ]:
i_range = np.linspace(0.50, 0.85, 100)
v_fixed2 = 0.50
spd_ref  = 3   # km/h

fig, ax = plt.subplots(figsize=(9, 5))

for (sname, rb), color in zip(services.items(), colors):
    ebn0 = ebn0_linear(sname, spd_ref)
    nm_vals = [nmax(rb, v_fixed2, ebn0, i=i_val) for i_val in i_range]
    ax.plot(i_range, nm_vals, color=color, linewidth=2, label=sname)

ax.axvspan(0.55, 0.80, alpha=0.10, color='gray', label='Rango típico i')
ax.set_xlabel('Factor de interferencia intercelular $i$')
ax.set_ylabel('$N_{max}$ [usuarios]')
ax.set_title(f'Sensibilidad de $N_{{max}}$ frente a i  (v={v_fixed2}, {spd_ref} km/h, η_target={eta_target})',
             fontweight='bold')
ax.legend(fontsize=9)
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.savefig('sensibilidad_i.png', bbox_inches='tight')
plt.show()
print("Figura guardada: sensibilidad_i.png")

---
## 8. Resumen de Resultados y Recomendación Operacional

### 8.1 Tabla resumen consolidada

In [ ]:
summary_rows = []
for sname, rb in services.items():
    for spd in speeds_kmh:
        for v_act in [0.40, 0.50, 0.70]:
            ebn0_lin = ebn0_linear(sname, spd)
            gp       = W_chips / rb
            nm       = nmax(rb, v_act, ebn0_lin)
            summary_rows.append({
                'Servicio':      sname,
                'Vel. [km/h]':   spd,
                'Act. v':        v_act,
                'Gp [dB]':       round(10*np.log10(gp), 1),
                'Eb/N0 [dB]':    ebn0_ref_dB[sname] + delta_ff_dB[spd],
                'Nmax':          int(nm),
            })

df_summary = pd.DataFrame(summary_rows)
print("Tabla consolidada de capacidad máxima:")
display(df_summary.to_string(index=False))

In [ ]:
# ── Heatmap de Nmax: servicio × velocidad (v=0.50) ──────────────────────────
pivot_data = df_summary[df_summary['Act. v'] == 0.50].pivot(
    index='Servicio', columns='Vel. [km/h]', values='Nmax'
)

fig, ax = plt.subplots(figsize=(7, 3.5))
im = ax.imshow(pivot_data.values, aspect='auto', cmap='RdYlGn',
               vmin=pivot_data.values.min(), vmax=pivot_data.values.max())

ax.set_xticks(range(len(speeds_kmh)))
ax.set_xticklabels([f'{s} km/h' for s in speeds_kmh], fontsize=11)
ax.set_yticks(range(len(services)))
ax.set_yticklabels(list(services.keys()), fontsize=10)
ax.set_title(f'Heatmap Nmax  (v=0.50, η_target={eta_target}, i={i_factor})',
             fontweight='bold')

for i_row in range(pivot_data.shape[0]):
    for j_col in range(pivot_data.shape[1]):
        ax.text(j_col, i_row, str(pivot_data.values[i_row, j_col]),
                ha='center', va='center', fontsize=13, fontweight='bold', color='black')

plt.colorbar(im, ax=ax, label='$N_{max}$', fraction=0.04)
plt.tight_layout()
plt.savefig('heatmap_nmax.png', bbox_inches='tight')
plt.show()
print("Figura guardada: heatmap_nmax.png")

---
## 9. Informe Operacional y Discusión Técnica

### 9.1 Recomendaciones para el gestor de red del campus

> **Contexto:** Nodo B WCDMA/UMTS FDD, $W = 3.84$ Mchip/s, zona exterior de campus con mezcla de peatones (3 km/h), usuarios en bicicleta (15 km/h) y vehículo de mantenimiento (50 km/h).

| Recomendación | Detalle |
|---|---|
| **R-1 Umbral operativo η ≤ 0.70** | Mantener la carga uplink por debajo del 70 % del polo de carga (≈ 5 dB de margen). Esto deja capacidad de emergencia y reduce la sensibilidad al ruido de control de potencia. |
| **R-2 Limitar sesiones de 64 kb/s en hora punta** | Una sesión de 64 kb/s consume ≈ 3–4 × el «slot de carga» de una sesión de voz. Si la carga supera el 60 %, bloquear nuevas sesiones de datos de alta tasa y ofrecerlas a las bandas LTE/5G disponibles. |
| **R-3 Monitorizar la velocidad media de los terminales** | A 50 km/h la $N_{max}$ cae ≈ 30–35 % respecto a 3 km/h por pérdida de eficacia del control de potencia. Si el paso de vehículos es frecuente, incrementar el margen de carga objetivo a 0.65. |
| **R-4 Ajustar el umbral de admisión según el mix de tráfico** | En hora punta con tráfico mixto (LMS, videollamada, sincronización), el factor de actividad medio sube hacia 0.65–0.70. Aplicar la regla $\hat{\eta} + \Delta\eta \leq 0.70$ en tiempo real. |
| **R-5 Calibrar periódicamente el bucle de control de potencia** | Un bucle mal calibrado equivale a un margen adicional de +2–3 dB en $E_b/N_0$, lo que reduce la capacidad un 20–25 %. Revisar el ajuste de los parámetros de control de potencia en el RNC cada semestre. |

### 9.2 ¿Por qué el uplink es más sensible que el downlink?

1. **Interferencia no coordinada:** En el uplink, cada terminal transmite con su propia potencia y el canal varía independientemente. La interferencia total en el NodoB es la suma de todas las contribuciones; cualquier terminal mal controlado eleva el nivel de ruido para *todos* los demás usuarios (efecto de ruido de acceso múltiple, MAI).

2. **Control de potencia limitado por la velocidad:** El bucle de control de potencia rápida (1 500 comandos/s en WCDMA) compensa el desvanecimiento de Rayleigh solo si la variación del canal es más lenta que el bucle. A 50 km/h y 2 GHz, la frecuencia Doppler máxima es ≈ 93 Hz, lo que satura el bucle y el Eb/N0 requerido crece +3.5 dB, reduciendo Nmax ≈ 30 %.

3. **Efecto de polo de carga:** La carga uplink acumula las contribuciones de todos los usuarios. Al acercarse a η = 1 (polo de carga), la interferencia total tiende a infinito y la célula colapsa. El downlink, controlado por la estación base, no tiene este polo en el mismo sentido (la potencia total está acotada por el amplificador).

4. **Mezcla de servicios:** En el uplink, un usuario de 64 kb/s consume la misma «ranura de interferencia» que ≈ 5 usuarios de voz AMR 12.2 kb/s. Esto hace que la mezcla de tráfico impacte directamente en Nmax y requiera un control de admisión más fino que en el downlink.

### 9.3 Conclusión numérica

Con los parámetros del campus ($W = 3.84$ Mchip/s, $i = 0.65$, $\eta_{target} = 0.70$, $v = 0.50$):

- **Voz AMR 12.2 kb/s, 3 km/h:** hasta ~**89 usuarios** simultáneos.
- **Datos 64 kb/s, 3 km/h:** hasta ~**39 usuarios** simultáneos.
- A **50 km/h**, ambos valores caen ≈ 30 %, subrayando la importancia de priorizar el uplink en zonas de tráfico rápido.

In [ ]:
# ── Verificación numérica de los valores de la conclusión ────────────────────
print("Verificación de valores clave de la conclusión:")
for sname, rb in services.items():
    for spd in [3, 50]:
        ebn0 = ebn0_linear(sname, spd)
        nm   = int(nmax(rb, 0.50, ebn0))
        print(f"  {sname:25s}  {spd:2d} km/h  →  Nmax = {nm}")